# EV vs ICE Training Pipeline

Train and compare multiple image classifiers to replace the **Stage 2 YOLO EV detector** in `Code/consolidated_dashcam_pipeline.ipynb`.

Includes the **current Stage 2 model** (`ev-car-vixid`) as two reference points:
1. **as-is** — evaluate existing weights with no extra training
2. **fine-tuned** — transfer-learn / fine-tune those weights on our labeled EV/ICE crops

Models are ranked primarily by **EV precision** (`test_precision_ev`).

Use `SELECTED_EXPERIMENTS` in the config cell to train/evaluate **one or a few models** without re-running the full matrix. Inference latency is measured separately from training time.

Post-YOLOv8 Ultralytics classifiers (`NEW_YOLO_EXPERIMENTS`: YOLO11 + YOLO26) are included for SOTA comparisons; selective runs can train only those and merge with prior `runs/*/result.json`.

## Data source

- Images: `dataset/ev/` and `dataset/ice/`
- Ground truth labels: `dataset/annotations.csv` → `label` column (`ev` / `ice`)
- Image paths: `relative_path` column (e.g. `ev/ev_1.jpg`)

Only rows with a valid image file and a label in `{ev, ice}` are used.

## Outputs

- Per-model metrics and comparison table under `runs/`
- Best model export under `models/best_ev_ice_classifier/` (`best.pt`, `metadata.json`, optional ONNX)
- Drop-in runtime wrapper: `ev_ice_classifier.py` → `LocalEVClassifier.score_crop(crop_bgr)`

## Pipeline integration

In `consolidated_dashcam_pipeline_best_proprietary_model.ipynb`, Stage 2 loads the export via `LocalEVClassifier`:

```python
from ev_ice_classifier import LocalEVClassifier

ev_classifier = LocalEVClassifier(
    model_dir=WORKSPACE_ROOT / 'Code/EV training module/Trainer/models/best_ev_ice_classifier',
    confidence_threshold=EV_CONFIDENCE_THRESHOLD,
)
```

`update_ev_classification(...)` and voting logic stay unchanged.


## 1) Environment and Imports

Prefer the `cv` conda environment used by the consolidated dashcam pipeline.

In [ ]:
# !pip install torch torchvision ultralytics opencv-python pandas numpy scikit-learn matplotlib seaborn tqdm onnx onnxscript


In [ ]:
from __future__ import annotations

import json
import random
import shutil
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm
from ultralytics import YOLO

sns.set_theme(style="whitegrid")


## 2) Configuration

Tweak these knobs to compare models and training settings.

**Selective runs:** set `SELECTED_EXPERIMENTS` to a list of experiment `name`s to run only those models.
Set it to `None` to run everything in `EXPERIMENTS`.

`NEW_YOLO_EXPERIMENTS` lists the post-YOLOv8 Ultralytics classifiers (YOLO11 + YOLO26). The next run is configured to train **only those** and compare against prior `runs/*/result.json` via `load_results_from_runs()`.

The first two experiments are the **current Stage 2** model from `consolidated_dashcam_pipeline.ipynb`:
- `vixid_vanilla` — evaluate existing weights only
- `vixid_transfer_learned` — transfer-learn those weights on our labeled crops

`SELECTION_METRIC = "f1_ev"` controls ranking, early stopping (torchvision), export selection, and comparison plots.



In [ ]:
TRAINER_ROOT = Path.cwd().resolve()
WORKSPACE_ROOT = TRAINER_ROOT.parents[2]
DATASET_ROOT = TRAINER_ROOT / "dataset"
ANNOTATIONS_CSV = DATASET_ROOT / "annotations.csv"
RUNS_DIR = TRAINER_ROOT / "runs"
EXPORT_DIR = TRAINER_ROOT / "models" / "best_ev_ice_classifier"

VALID_LABELS = ("ev", "ice")
CLASS_TO_IDX = {"ev": 0, "ice": 1}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
EV_INDEX = CLASS_TO_IDX["ev"]

STAGE2_MODEL_DIR = WORKSPACE_ROOT / "Code/EV experiement/models/ev-car-vixid"
STAGE2_WEIGHTS_CANDIDATE = STAGE2_MODEL_DIR / "weights/best.pt"
STAGE2_CONFIDENCE_THRESHOLD = 0.70
STAGE2_INFERENCE_CONFIDENCE = 0.25
STAGE2_NON_EV_CLASS_NAMES = {"Non_EV", "Non-EV", "NON-EV", "MG (Non-EV)", "ice"}
STAGE2_FALLBACK_PRETRAIN = "yolov8n.pt"

SEED = 42
VAL_SIZE = 0.20
TEST_SIZE = 0.15
NUM_WORKERS = 0
EARLY_STOPPING_PATIENCE = 8

EXPERIMENTS: list[dict] = [
    {
        "name": "vixid_vanilla",
        "family": "yolo_cls",
        "backbone": "vixid_weights",
        "img_size": 640,
        "batch_size": 1,
        "epochs": 0,
        "lr": 0.0,
        "weight_decay": 0.0,
        "freeze_backbone_epochs": 0,
        "use_class_weights": False,
        "strong_augment": False,
        "description": (
            "Vixid Stage-2 EV model evaluated as-is — no training (vixid_vanilla). "
            "Uses the same LocalYOLOEVClassifier scoring rules as consolidated_dashcam_pipeline "
            "(EV brand conf vs Non_EV, threshold=0.70). Primary reference baseline."
        ),
    },
    {
        "name": "vixid_transfer_learned",
        "family": "yolo_cls",
        "backbone": "vixid_weights",
        "img_size": 640,
        "batch_size": 8,
        "epochs": 40,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "Transfer-learn the Vixid Stage-2 YOLOv8n EV detector on our labeled EV/ICE crops "
            "(vixid_transfer_learned). Builds a binary detection dataset (ev / Non_EV) and fine-tunes "
            "from Stage-2 .pt weights when available. Closest apples-to-apples upgrade path."
        ),
    },
    {
        "name": "resnet18_baseline",
        "family": "torchvision",
        "backbone": "resnet18",
        "img_size": 224,
        "batch_size": 16,
        "epochs": 30,
        "lr": 1e-4,
        "weight_decay": 1e-4,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "ResNet-18 baseline (~11M params). Fast, lightweight sanity-check model. "
            "ImageNet-pretrained; trains end-to-end from epoch 1. Good starting point "
            "before investing in heavier architectures. Exported via ev_ice_classifier.py."
        ),
    },
    {
        "name": "resnet50_heavy",
        "family": "torchvision",
        "backbone": "resnet50",
        "img_size": 224,
        "batch_size": 12,
        "epochs": 30,
        "lr": 5e-5,
        "weight_decay": 1e-4,
        "freeze_backbone_epochs": 2,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "ResNet-50 (~25M params). Higher capacity for subtle EV vs ICE cues. "
            "Backbone frozen for 2 epochs (head-only warmup), then full fine-tune "
            "at lower lr (5e-5). Slower but often best torchvision performer on small sets."
        ),
    },
    {
        "name": "efficientnet_b0_fast",
        "family": "torchvision",
        "backbone": "efficientnet_b0",
        "img_size": 224,
        "batch_size": 16,
        "epochs": 30,
        "lr": 1e-4,
        "weight_decay": 1e-4,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "EfficientNet-B0 (~5M params). Efficient architecture — strong accuracy-to-"
            "speed ratio. Same training recipe as ResNet-18. Useful if ResNet-50 is "
            "too slow but you want more capacity than MobileNet."
        ),
    },
    {
        "name": "mobilenet_v3_small_edge",
        "family": "torchvision",
        "backbone": "mobilenet_v3_small",
        "img_size": 224,
        "batch_size": 24,
        "epochs": 30,
        "lr": 1e-4,
        "weight_decay": 1e-4,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": False,
        "description": (
            "MobileNetV3-Small (~2.5M params). Smallest/fastest torchvision option — "
            "ideal for edge deployment or CPU inference in the dashcam pipeline. "
            "Milder augmentation (strong_augment=False) to reduce noise on tiny datasets."
        ),
    },

    {
        "name": "vit_b16",
        "family": "torchvision",
        "backbone": "vit_b_16",
        "img_size": 224,
        "batch_size": 8,
        "epochs": 25,
        "lr": 3e-5,
        "weight_decay": 1e-4,
        "freeze_backbone_epochs": 2,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "ViT-B/16 (~86M params). Vision Transformer baseline — attention over image patches. "
            "Strong ImageNet features; higher capacity/latency than ResNet-50. Freeze backbone "
            "2 epochs then fine-tune at low lr. Exported via ev_ice_classifier.py."
        ),
    },
    {
        "name": "vit_b32",
        "family": "torchvision",
        "backbone": "vit_b_32",
        "img_size": 224,
        "batch_size": 12,
        "epochs": 25,
        "lr": 3e-5,
        "weight_decay": 1e-4,
        "freeze_backbone_epochs": 2,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "ViT-B/32 (~88M params). Coarser 32px patches than B/16 — typically faster inference "
            "with a small accuracy trade-off. Same freeze/fine-tune recipe as vit_b16."
        ),
    },
    {
        "name": "yolov8n_cls",
        "family": "yolo_cls",
        "backbone": "yolov8n-cls.pt",
        "img_size": 224,
        "batch_size": 16,
        "epochs": 40,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "YOLOv8n-cls (nano). Ultralytics image classifier — same library as Stage 1. "
            "Fastest YOLO-cls variant. Trained via model.train() on a folder-based dataset. "
            "Note: not directly compatible with ev_ice_classifier.py export (torchvision only)."
        ),
    },
    {
        "name": "yolov8s_cls",
        "family": "yolo_cls",
        "backbone": "yolov8s-cls.pt",
        "img_size": 224,
        "batch_size": 12,
        "epochs": 40,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "YOLOv8s-cls (small). More capacity than nano YOLO-cls while staying fast. "
            "Tests whether a slightly larger YOLO classifier outperforms torchvision models "
            "on padded dashcam vehicle crops. Same export/integration caveats as yolov8n_cls."
        ),
    },

    {
        "name": "yolo11n_cls",
        "family": "yolo_cls",
        "backbone": "yolo11n-cls.pt",
        "img_size": 224,
        "batch_size": 16,
        "epochs": 40,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "YOLO11n-cls (nano). Successor to YOLOv8n-cls — stronger ImageNet baseline at "
            "similar latency. Direct SOTA drop-in comparison against yolov8n_cls."
        ),
    },
    {
        "name": "yolo11s_cls",
        "family": "yolo_cls",
        "backbone": "yolo11s-cls.pt",
        "img_size": 224,
        "batch_size": 12,
        "epochs": 40,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "YOLO11s-cls (small). More capacity than YOLO11n; peer comparison for yolov8s_cls."
        ),
    },
    {
        "name": "yolo11m_cls",
        "family": "yolo_cls",
        "backbone": "yolo11m-cls.pt",
        "img_size": 224,
        "batch_size": 8,
        "epochs": 40,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "YOLO11m-cls (medium). Higher-capacity YOLO11 classifier for accuracy ceiling vs "
            "nano/small and torchvision baselines. Lower batch size for memory."
        ),
    },

    {
        "name": "yolo26n_cls",
        "family": "yolo_cls",
        "backbone": "yolo26n-cls.pt",
        "img_size": 224,
        "batch_size": 16,
        "epochs": 40,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "YOLO26n-cls (nano). Latest Ultralytics classify release — SOTA nano reference "
            "against yolov8n_cls / yolo11n_cls on EV vs ICE crops."
        ),
    },
    {
        "name": "yolo26s_cls",
        "family": "yolo_cls",
        "backbone": "yolo26s-cls.pt",
        "img_size": 224,
        "batch_size": 12,
        "epochs": 40,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "YOLO26s-cls (small). Latest-generation small classifier for SOTA speed/accuracy "
            "trade-off vs yolov8s_cls / yolo11s_cls."
        ),
    },
    {
        "name": "yolo26m_cls",
        "family": "yolo_cls",
        "backbone": "yolo26m-cls.pt",
        "img_size": 224,
        "batch_size": 8,
        "epochs": 40,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "freeze_backbone_epochs": 0,
        "use_class_weights": True,
        "strong_augment": True,
        "description": (
            "YOLO26m-cls (medium). Highest-capacity new YOLO in this matrix — accuracy ceiling "
            "for Ultralytics classify on this dataset (before x-large)."
        ),
    },
]

SELECTED_EXPERIMENTS: list[str] | None = [
    "yolo11n_cls",
    "yolo26n_cls",
    "yolo11s_cls",
    "yolo26s_cls",
    "yolo11m_cls",
    "yolo26m_cls",
]
# SELECTED_EXPERIMENTS = None

LATENCY_WARMUP = 10
LATENCY_MAX_IMAGES = None

K_FOLD = 5

EXPORT_ONNX = True
SELECTION_METRIC = "f1_ev"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.parent.mkdir(parents=True, exist_ok=True)

print(f"Trainer root : {TRAINER_ROOT}")
print(f"Workspace    : {WORKSPACE_ROOT}")
print(f"Dataset root : {DATASET_ROOT}")
print(f"Stage-2 dir  : {STAGE2_MODEL_DIR}")
print(f"Rank metric  : {SELECTION_METRIC}")
print(f"Experiments  : {len(EXPERIMENTS)}")
if SELECTED_EXPERIMENTS is None:
    print("Selected     : ALL")
else:
    print(f"Selected     : {SELECTED_EXPERIMENTS}")
print()
for i, cfg in enumerate(EXPERIMENTS, start=1):
    marker = "*" if (SELECTED_EXPERIMENTS is None or cfg["name"] in SELECTED_EXPERIMENTS) else " "
    print(f"{marker}[{i}] {cfg['name']} ({cfg['family']} / {cfg['backbone']})")
    print(f"     {cfg.get('description', 'No description provided.')}")
    print()


## 3) Load and Validate Dataset

Labels come from `annotations.csv` (not folder names). Rows with missing files are dropped.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def load_labeled_dataset(csv_path: Path, dataset_root: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path).copy()
    required = {"relative_path", "label"}
    missing_cols = required - set(df.columns)
    if missing_cols:
        raise ValueError(f"annotations.csv missing columns: {sorted(missing_cols)}")

    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df = df[df["label"].isin(VALID_LABELS)].copy()
    df["image_path"] = df["relative_path"].map(lambda rel: dataset_root / rel)
    df["exists"] = df["image_path"].map(lambda p: Path(p).exists())

    missing = df[~df["exists"]]
    if len(missing):
        print(f"Dropping {len(missing)} rows with missing image files:")
        display(missing[["relative_path", "label"]].head(10))

    df = df[df["exists"]].reset_index(drop=True)
    df["label_idx"] = df["label"].map(CLASS_TO_IDX).astype(int)
    return df

set_seed(SEED)
DEVICE = resolve_device()
print(f"Device: {DEVICE}")

labeled_df = load_labeled_dataset(ANNOTATIONS_CSV, DATASET_ROOT)
print(f"Usable labeled images: {len(labeled_df)}")
print(labeled_df["label"].value_counts())

if len(labeled_df) < 20:
    print(
        "\nWARNING: Very small labeled set. Metrics will be noisy. "
        "Label more images in Vehicle Tinder, then re-run."
    )

labeled_df.head()


In [ ]:
_preview_train, _preview_test = train_test_split(
    labeled_df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=labeled_df["label_idx"],
)
_val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
_preview_train, _preview_val = train_test_split(
    _preview_train,
    test_size=_val_ratio,
    random_state=SEED,
    stratify=_preview_train["label_idx"],
)

split_counts = pd.DataFrame(
    {
        "train": _preview_train["label"].value_counts(),
        "val": _preview_val["label"].value_counts(),
        "test": _preview_test["label"].value_counts(),
    }
).fillna(0).astype(int).reindex(["ev", "ice"])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
labeled_df["label"].value_counts().reindex(["ev", "ice"]).plot(
    kind="bar", ax=ax[0], color=["#7b2cbf", "#4c6ef5"]
)
ax[0].set_title("Label distribution")
ax[0].set_xlabel("label")

split_counts.T.plot(kind="bar", stacked=True, ax=ax[1], color=["#7b2cbf", "#4c6ef5"])
ax[1].set_title("Train / val / test counts")
ax[1].set_xlabel("split")
ax[1].set_ylabel("images")
ax[1].legend(title="label")
for i, total in enumerate(split_counts.sum(axis=0)):
    ax[1].text(i, total + max(1, total * 0.02), str(int(total)), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print(split_counts)
print("Totals:", {k: int(v) for k, v in split_counts.sum(axis=0).items()})

sample = labeled_df.groupby("label", group_keys=False).apply(
    lambda g: g.sample(min(3, len(g)), random_state=SEED)
)
fig2, axes = plt.subplots(2, 3, figsize=(10, 6))
for ax_i, (_, row) in zip(axes.flatten(), sample.iterrows()):
    img = cv2.cvtColor(cv2.imread(str(row["image_path"])), cv2.COLOR_BGR2RGB)
    ax_i.imshow(img)
    ax_i.set_title(f"{row['label']} | {row['filename']}")
    ax_i.axis("off")
plt.tight_layout()
plt.show()


## 4) Dataset, Augmentations, and Model Factory

In [ ]:
class VehicleCropDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform: transforms.Compose | None = None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        image = cv2.imread(str(row["image_path"]))
        if image is None:
            raise FileNotFoundError(row["image_path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = int(row["label_idx"])
        if self.transform is not None:
            image = self.transform(image)
        return image, label

def build_transforms(img_size: int, strong_augment: bool, train: bool) -> transforms.Compose:
    if not train:
        return transforms.Compose(
            [
                transforms.ToPILImage(),
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]
        )

    aug: list = [
        transforms.ToPILImage(),
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
    ]
    if strong_augment:
        aug.extend(
            [
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
                transforms.RandomAffine(degrees=8, translate=(0.05, 0.05), scale=(0.9, 1.1)),
            ]
        )
    aug.extend(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    return transforms.Compose(aug)

def build_torchvision_model(backbone: str, num_classes: int = 2) -> nn.Module:
    name = backbone.lower()
    weights = models.ResNet18_Weights.DEFAULT

    if name == "resnet18":
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif name == "mobilenet_v3_small":
        model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    elif name == "vit_b_16":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    elif name == "vit_b_32":
        model = models.vit_b_32(weights=models.ViT_B_32_Weights.DEFAULT)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    else:
        raise ValueError(f"Unsupported torchvision backbone: {backbone}")
    return model

def set_backbone_trainable(model: nn.Module, trainable: bool) -> None:
    for param in model.parameters():
        param.requires_grad = trainable
    if trainable:
        for param in model.parameters():
            param.requires_grad = True

def compute_class_weights(frame: pd.DataFrame, device: torch.device) -> torch.Tensor:
    counts = frame["label_idx"].value_counts().sort_index()
    total = counts.sum()
    weights = total / (len(counts) * counts)
    return torch.tensor(weights.values, dtype=torch.float32, device=device)


## 5) Train / Evaluate Helpers

In [ ]:
@dataclass
class EvalMetrics:
    accuracy: float
    precision_ev: float
    recall_ev: float
    f1_ev: float
    roc_auc: float | None
    report: str
    y_true: list[int]
    y_pred: list[int]
    y_prob_ev: list[float]

def evaluate_torchvision_model(model: nn.Module, loader: DataLoader, device: torch.device) -> EvalMetrics:
    model.eval()
    y_true: list[int] = []
    y_pred: list[int] = []
    y_prob_ev: list[float] = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())
            y_prob_ev.extend(probs[:, EV_INDEX].cpu().tolist())

    return _metrics_from_arrays(y_true, y_pred, y_prob_ev)

def _metrics_from_arrays(y_true: list[int], y_pred: list[int], y_prob_ev: list[float]) -> EvalMetrics:
    report = classification_report(
        y_true,
        y_pred,
        target_names=[IDX_TO_CLASS[i] for i in sorted(IDX_TO_CLASS)],
        digits=4,
        zero_division=0,
    )
    roc = None
    if len(set(y_true)) > 1:
        y_true_ev = [1 if y == EV_INDEX else 0 for y in y_true]
        roc = float(roc_auc_score(y_true_ev, y_prob_ev))

    return EvalMetrics(
        accuracy=float(accuracy_score(y_true, y_pred)),
        precision_ev=float(precision_score(y_true, y_pred, pos_label=EV_INDEX, zero_division=0)),
        recall_ev=float(recall_score(y_true, y_pred, pos_label=EV_INDEX, zero_division=0)),
        f1_ev=float(f1_score(y_true, y_pred, pos_label=EV_INDEX, zero_division=0)),
        roc_auc=roc,
        report=report,
        y_true=y_true,
        y_pred=y_pred,
        y_prob_ev=y_prob_ev,
    )

def train_torchvision_experiment(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    cfg: dict,
    run_dir: Path,
) -> tuple[nn.Module, EvalMetrics, dict]:
    run_dir.mkdir(parents=True, exist_ok=True)

    train_tf = build_transforms(cfg["img_size"], cfg["strong_augment"], train=True)
    val_tf = build_transforms(cfg["img_size"], cfg["strong_augment"], train=False)

    train_loader = DataLoader(
        VehicleCropDataset(train_df, train_tf),
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=NUM_WORKERS,
    )
    val_loader = DataLoader(
        VehicleCropDataset(val_df, val_tf),
        batch_size=cfg["batch_size"],
        shuffle=False,
        num_workers=NUM_WORKERS,
    )

    model = build_torchvision_model(cfg["backbone"]).to(DEVICE)
    if cfg.get("freeze_backbone_epochs", 0) > 0:
        for name, param in model.named_parameters():
            if (
                name.startswith("fc.")
                or name.startswith("classifier.")
                or name.startswith("heads.")
            ):
                param.requires_grad = True
            else:
                param.requires_grad = False

    criterion = nn.CrossEntropyLoss(
        weight=compute_class_weights(train_df, DEVICE) if cfg.get("use_class_weights", True) else None
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_metrics: EvalMetrics | None = None
    best_state = None
    stale_epochs = 0
    history: list[dict] = []

    for epoch in range(1, cfg["epochs"] + 1):
        if epoch == cfg.get("freeze_backbone_epochs", 0) + 1 and cfg.get("freeze_backbone_epochs", 0) > 0:
            set_backbone_trainable(model, True)
            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=cfg["lr"],
                weight_decay=cfg["weight_decay"],
            )

        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item())

        metrics = evaluate_torchvision_model(model, val_loader, DEVICE)
        history.append(
            {
                "epoch": epoch,
                "train_loss": running_loss / max(1, len(train_loader)),
                "val_accuracy": metrics.accuracy,
                "val_precision_ev": metrics.precision_ev,
                "val_f1_ev": metrics.f1_ev,
            }
        )

        improved = best_metrics is None or metrics.precision_ev > best_metrics.precision_ev
        if improved:
            best_metrics = metrics
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
            torch.save(best_state, run_dir / "best.pt")
        else:
            stale_epochs += 1

        print(
            f"Epoch {epoch:02d}/{cfg['epochs']} | loss={history[-1]['train_loss']:.4f} "
            f"| val_acc={metrics.accuracy:.3f} | val_prec_ev={metrics.precision_ev:.3f} "
            f"| val_f1_ev={metrics.f1_ev:.3f}"
        )

        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    assert best_metrics is not None
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    (run_dir / "classification_report.txt").write_text(best_metrics.report)
    return model, best_metrics, {"history": history}


In [ ]:
def write_yolo_cls_dataset(frame: pd.DataFrame, out_dir: Path) -> Path:
    """Create an Ultralytics classification dataset folder from labeled rows."""
    if out_dir.exists():
        shutil.rmtree(out_dir)
    for split_name, split_df in frame.groupby("split"):
        for _, row in split_df.iterrows():
            src = Path(row["image_path"])
            dst = out_dir / str(split_name) / str(row["label"]) / src.name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    return out_dir

def train_yolo_cls_experiment(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    cfg: dict,
    run_dir: Path,
) -> tuple[Path, EvalMetrics, dict]:
    run_dir.mkdir(parents=True, exist_ok=True)
    yolo_df = pd.concat(
        [
            train_df.assign(split="train"),
            val_df.assign(split="val"),
        ],
        ignore_index=True,
    )
    yolo_root = write_yolo_cls_dataset(yolo_df, run_dir / "yolo_dataset")

    model = YOLO(cfg["backbone"])
    results = model.train(
        data=str(yolo_root),
        epochs=cfg["epochs"],
        imgsz=cfg["img_size"],
        batch=cfg["batch_size"],
        lr0=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        project=str(run_dir),
        name="yolo_train",
        exist_ok=True,
        verbose=False,
        seed=SEED,
        patience=EARLY_STOPPING_PATIENCE,
        device=str(DEVICE),
    )

    best_weights = Path(results.save_dir) / "weights" / "best.pt"
    if not best_weights.exists():
        raise FileNotFoundError(f"YOLO training did not produce weights at {best_weights}")

    yolo_model = YOLO(str(best_weights))
    y_true: list[int] = []
    y_pred: list[int] = []
    y_prob_ev: list[float] = []

    for _, row in val_df.iterrows():
        result = yolo_model.predict(str(row["image_path"]), verbose=False)[0]
        probs = result.probs
        if probs is None:
            continue
        names = yolo_model.names
        pred_idx = int(probs.top1)
        pred_label = str(names[pred_idx] if isinstance(names, dict) else names[pred_idx]).lower()
        pred_idx_mapped = CLASS_TO_IDX.get(pred_label, pred_idx)

        y_true.append(int(row["label_idx"]))
        y_pred.append(pred_idx_mapped)
        prob_vec = probs.data.cpu().numpy()
        ev_idx = CLASS_TO_IDX["ev"]
        y_prob_ev.append(float(prob_vec[ev_idx]) if ev_idx < len(prob_vec) else float(probs.top1conf))

    metrics = _metrics_from_arrays(y_true, y_pred, y_prob_ev)
    shutil.copy2(best_weights, run_dir / "best.pt")
    (run_dir / "classification_report.txt").write_text(metrics.report)
    return best_weights, metrics, {"yolo_save_dir": str(results.save_dir)}

def _normalize_class_name(name: str) -> str:
    return str(name).strip().replace(" ", "_")

def find_stage2_weights(search_root: Path) -> Path | None:
    """Locate Stage-2 checkpoint (.pt preferred, then .onnx), matching the pipeline."""
    if not search_root.exists():
        return None
    for name in ("best.pt", "weights.pt", "weights.onnx"):
        matches = sorted(search_root.rglob(name))
        if matches:
            return matches[0]
    pt_matches = sorted(search_root.rglob("*.pt"))
    if pt_matches:
        return pt_matches[0]
    onnx_matches = sorted(search_root.rglob("*.onnx"))
    if onnx_matches:
        return onnx_matches[0]
    return None

def resolve_stage2_inference_weights() -> Path:
    """Resolve weights usable for as-is inference (.pt or .onnx)."""
    if STAGE2_WEIGHTS_CANDIDATE.exists():
        return STAGE2_WEIGHTS_CANDIDATE
    found = find_stage2_weights(STAGE2_MODEL_DIR)
    if found is None:
        raise FileNotFoundError(
            f"Stage-2 EV weights not found under {STAGE2_MODEL_DIR}. "
            "Place best.pt / weights.onnx there, or download via the consolidated pipeline."
        )
    print(f"Resolved Stage-2 inference weights: {found}")
    return found

def resolve_stage2_train_weights() -> Path:
    """Resolve .pt weights for fine-tuning; fall back to yolov8n.pt if only ONNX exists."""
    candidate = STAGE2_WEIGHTS_CANDIDATE
    if candidate.exists() and candidate.suffix.lower() == ".pt":
        return candidate
    found = find_stage2_weights(STAGE2_MODEL_DIR)
    if found is not None and found.suffix.lower() == ".pt":
        print(f"Using Stage-2 .pt for fine-tuning: {found}")
        return found
    if found is not None:
        print(
            f"WARNING: Stage-2 fine-tune needs .pt weights, but only found {found}. "
            f"Falling back to {STAGE2_FALLBACK_PRETRAIN} (architecture match: YOLOv8n). "
            "Place exported best.pt under STAGE2_MODEL_DIR/weights/ for true transfer learning."
        )
    else:
        print(
            f"WARNING: No Stage-2 weights found. Fine-tuning from {STAGE2_FALLBACK_PRETRAIN}."
        )
    return Path(STAGE2_FALLBACK_PRETRAIN)

@dataclass
class Stage2YOLODetectorScorer:
    """Mirrors LocalYOLOEVClassifier.score_crop from consolidated_dashcam_pipeline."""

    weights_path: str | Path
    confidence_threshold: float = STAGE2_CONFIDENCE_THRESHOLD
    inference_confidence: float = STAGE2_INFERENCE_CONFIDENCE
    non_ev_classes: set[str] | None = None
    model: object = None

    def __post_init__(self) -> None:
        if self.non_ev_classes is None:
            self.non_ev_classes = set(STAGE2_NON_EV_CLASS_NAMES)
        self.non_ev_classes = {_normalize_class_name(c) for c in self.non_ev_classes}
        self.weights_path = str(self.weights_path)
        print(f"Loading Stage-2 YOLO detector: {self.weights_path}")
        self.model = YOLO(self.weights_path)
        names = self.model.names
        class_list = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
        print(f"Stage-2 classes ({len(class_list)}): {class_list}")

    def _is_non_ev(self, class_name: str) -> bool:
        normalized = _normalize_class_name(class_name)
        if normalized in self.non_ev_classes:
            return True
        return "non" in normalized.lower() and "ev" in normalized.lower()

    def _class_name_for_id(self, cls_id: int) -> str:
        names = self.model.names if self.model is not None else {}
        if isinstance(names, dict):
            return str(names.get(cls_id, cls_id))
        return str(names[cls_id])

    def score_crop(self, crop_bgr: np.ndarray) -> tuple[float, str | None]:
        """Return (ev_confidence, ev_class_or_None) using pipeline rules."""
        results = self.model.predict(crop_bgr, conf=self.inference_confidence, verbose=False)
        result = results[0]
        if result.boxes is None or len(result.boxes) == 0:
            return 0.0, None

        best_ev_conf = 0.0
        best_ev_class: str | None = None
        best_non_ev_conf = 0.0

        for box in result.boxes:
            cls_id = int(box.cls[0].item())
            class_name = self._class_name_for_id(cls_id)
            conf = float(box.conf[0].item())
            if self._is_non_ev(class_name):
                best_non_ev_conf = max(best_non_ev_conf, conf)
            elif conf > best_ev_conf:
                best_ev_conf = conf
                best_ev_class = class_name

        if best_ev_conf < self.confidence_threshold:
            return 0.0, None
        if best_non_ev_conf > best_ev_conf:
            return 0.0, None
        return best_ev_conf, best_ev_class

    def score_crop_raw(self, crop_bgr: np.ndarray) -> float:
        """EV confidence before hard threshold (for ROC); 0 if Non_EV dominates."""
        results = self.model.predict(crop_bgr, conf=self.inference_confidence, verbose=False)
        result = results[0]
        if result.boxes is None or len(result.boxes) == 0:
            return 0.0
        best_ev_conf = 0.0
        best_non_ev_conf = 0.0
        for box in result.boxes:
            cls_id = int(box.cls[0].item())
            class_name = self._class_name_for_id(cls_id)
            conf = float(box.conf[0].item())
            if self._is_non_ev(class_name):
                best_non_ev_conf = max(best_non_ev_conf, conf)
            else:
                best_ev_conf = max(best_ev_conf, conf)
        if best_non_ev_conf > best_ev_conf:
            return 0.0
        return best_ev_conf

def evaluate_stage2_scorer(scorer: Stage2YOLODetectorScorer, frame: pd.DataFrame) -> EvalMetrics:
    y_true: list[int] = []
    y_pred: list[int] = []
    y_prob_ev: list[float] = []
    for _, row in frame.iterrows():
        crop = cv2.imread(str(row["image_path"]))
        if crop is None:
            continue
        ev_conf, ev_label = scorer.score_crop(crop)
        raw_conf = scorer.score_crop_raw(crop)
        y_true.append(int(row["label_idx"]))
        y_pred.append(EV_INDEX if ev_label is not None else CLASS_TO_IDX["ice"])
        y_prob_ev.append(float(raw_conf))
    return _metrics_from_arrays(y_true, y_pred, y_prob_ev)

def evaluate_stage2_as_is(eval_df: pd.DataFrame, run_dir: Path) -> tuple[Path, EvalMetrics, dict]:
    """Evaluate the production Stage-2 detector with no training."""
    run_dir.mkdir(parents=True, exist_ok=True)
    weights = resolve_stage2_inference_weights()
    scorer = Stage2YOLODetectorScorer(weights_path=weights)
    metrics = evaluate_stage2_scorer(scorer, eval_df)
    artifact = run_dir / ("weights" + Path(weights).suffix)
    try:
        if Path(weights).resolve() != artifact.resolve():
            shutil.copy2(weights, artifact)
    except Exception:
        artifact = Path(weights)
    (run_dir / "classification_report.txt").write_text(metrics.report)
    with open(run_dir / "stage2_weights.json", "w") as f:
        json.dump({"weights": str(weights), "mode": "as_is"}, f, indent=2)
    return Path(weights), metrics, {"weights": str(weights), "mode": "as_is"}

def write_yolo_det_dataset_from_crops(frame: pd.DataFrame, out_dir: Path) -> Path:
    """Build a YOLO detection dataset from whole-crop labels (one box per image)."""
    if out_dir.exists():
        shutil.rmtree(out_dir)
    names = ["ev", "Non_EV"]
    label_to_cls = {"ev": 0, "ice": 1}
    for split_name, split_df in frame.groupby("split"):
        img_dir = out_dir / "images" / str(split_name)
        lbl_dir = out_dir / "labels" / str(split_name)
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        for _, row in split_df.iterrows():
            src = Path(row["image_path"])
            dst = img_dir / src.name
            shutil.copy2(src, dst)
            cls_id = label_to_cls[str(row["label"]).lower()]
            (lbl_dir / f"{src.stem}.txt").write_text(f"{cls_id} 0.5 0.5 0.95 0.95\n")
    data_yaml = out_dir / "data.yaml"
    data_yaml.write_text(
        "path: {path}\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n"
        "  0: ev\n"
        "  1: Non_EV\n".format(path=str(out_dir.resolve()))
    )
    return data_yaml

def train_stage2_finetune_experiment(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    cfg: dict,
    run_dir: Path,
) -> tuple[Path, EvalMetrics, dict]:
    """Fine-tune Stage-2 YOLO detector on labeled EV/ICE crops."""
    run_dir.mkdir(parents=True, exist_ok=True)
    train_weights = resolve_stage2_train_weights()
    yolo_df = pd.concat(
        [train_df.assign(split="train"), val_df.assign(split="val")],
        ignore_index=True,
    )
    data_yaml = write_yolo_det_dataset_from_crops(yolo_df, run_dir / "yolo_det_dataset")

    model = YOLO(str(train_weights))
    results = model.train(
        data=str(data_yaml),
        epochs=cfg["epochs"],
        imgsz=cfg["img_size"],
        batch=cfg["batch_size"],
        lr0=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        project=str(run_dir),
        name="stage2_finetune",
        exist_ok=True,
        verbose=False,
        seed=SEED,
        patience=EARLY_STOPPING_PATIENCE,
        device=str(DEVICE),
    )
    best_weights = Path(results.save_dir) / "weights" / "best.pt"
    if not best_weights.exists():
        raise FileNotFoundError(f"Stage-2 fine-tune did not produce weights at {best_weights}")

    scorer = Stage2YOLODetectorScorer(weights_path=best_weights)
    metrics = evaluate_stage2_scorer(scorer, val_df)
    shutil.copy2(best_weights, run_dir / "best.pt")
    (run_dir / "classification_report.txt").write_text(metrics.report)
    return best_weights, metrics, {
        "yolo_save_dir": str(results.save_dir),
        "pretrained_from": str(train_weights),
        "mode": "finetune",
    }


## 6) Run Experiments (modular)

1. **Prepare** train/val/test splits (next cell) — run once.
2. **Runner API** — defines `run_experiment_by_name(...)` and latency measurement.
3. **Select & run** — edit `SELECTED_EXPERIMENTS` (or call `run_experiment_by_name` for one model).
4. **Compare** — builds `comparison_df` with metrics + **true inference latency**.

Inference latency is measured with warmup + timed forward passes on test crops.
It does **not** include training time (the old `elapsed_sec / n_test` estimate was wrong and also misaligned after sorting).



In [ ]:
def split_dataframe(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=df["label_idx"],
    )
    val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
    train_df, val_df = train_test_split(
        train_df,
        test_size=val_ratio,
        random_state=SEED,
        stratify=train_df["label_idx"],
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

train_df, val_df, test_df = split_dataframe(labeled_df)
EXPERIMENT_BY_NAME = {cfg["name"]: cfg for cfg in EXPERIMENTS}
if "experiment_results" not in globals():
    experiment_results: list[dict] = []

print(f"Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")
print(train_df["label"].value_counts())
print(val_df["label"].value_counts())
print(test_df["label"].value_counts())
print(f"\nAvailable experiments ({len(EXPERIMENT_BY_NAME)}):")
for name in EXPERIMENT_BY_NAME:
    print(f"  - {name}")
print(f"\nCached results in memory: {len(experiment_results)}")

def _sync_device() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        try:
            torch.mps.synchronize()
        except Exception:
            pass

def _latency_stats(times_ms: list[float]) -> dict:
    arr = np.asarray(times_ms, dtype=np.float64)
    return {
        "infer_ms_mean": float(arr.mean()) if len(arr) else float("nan"),
        "infer_ms_median": float(np.median(arr)) if len(arr) else float("nan"),
        "infer_ms_p95": float(np.percentile(arr, 95)) if len(arr) else float("nan"),
        "infer_images": int(len(arr)),
    }

def measure_torchvision_latency(model: nn.Module, frame: pd.DataFrame, cfg: dict) -> dict:
    """Pure forward-pass latency on BGR crops (batch size 1)."""
    model.eval()
    tfm = build_transforms(cfg["img_size"], False, train=False)
    paths = list(frame["image_path"])
    if LATENCY_MAX_IMAGES is not None:
        paths = paths[: int(LATENCY_MAX_IMAGES)]

    def _one(path: Path) -> None:
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        tensor = tfm(img).unsqueeze(0).to(DEVICE)
        _sync_device()
        with torch.no_grad():
            _ = model(tensor)
        _sync_device()

    for path in paths[:LATENCY_WARMUP]:
        _one(path)

    times_ms: list[float] = []
    for path in paths:
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        tensor = tfm(img).unsqueeze(0).to(DEVICE)
        _sync_device()
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model(tensor)
        _sync_device()
        times_ms.append((time.perf_counter() - t0) * 1000.0)
    return _latency_stats(times_ms)

def measure_stage2_latency(scorer: Stage2YOLODetectorScorer, frame: pd.DataFrame) -> dict:
    """Pure score_crop latency — same call path as the dashcam pipeline."""
    paths = list(frame["image_path"])
    if LATENCY_MAX_IMAGES is not None:
        paths = paths[: int(LATENCY_MAX_IMAGES)]

    for path in paths[:LATENCY_WARMUP]:
        crop = cv2.imread(str(path))
        if crop is not None:
            scorer.score_crop(crop)

    times_ms: list[float] = []
    for path in paths:
        crop = cv2.imread(str(path))
        if crop is None:
            continue
        t0 = time.perf_counter()
        scorer.score_crop(crop)
        times_ms.append((time.perf_counter() - t0) * 1000.0)
    stats = _latency_stats(times_ms)
    stats["infer_backend"] = Path(str(scorer.weights_path)).suffix.lower().lstrip(".")
    return stats

def measure_yolo_cls_latency(yolo_model: YOLO, frame: pd.DataFrame) -> dict:
    paths = list(frame["image_path"])
    if LATENCY_MAX_IMAGES is not None:
        paths = paths[: int(LATENCY_MAX_IMAGES)]

    for path in paths[:LATENCY_WARMUP]:
        yolo_model.predict(str(path), verbose=False)

    times_ms: list[float] = []
    for path in paths:
        t0 = time.perf_counter()
        yolo_model.predict(str(path), verbose=False)
        times_ms.append((time.perf_counter() - t0) * 1000.0)
    return _latency_stats(times_ms)

def run_single_experiment(
    cfg: dict,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> dict:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    run_dir = RUNS_DIR / f"{cfg['name']}_{stamp}"
    started = time.time()
    latency: dict = {}

    if cfg["family"] == "torchvision":
        model, val_metrics, extra = train_torchvision_experiment(train_df, val_df, cfg, run_dir)
        test_loader = DataLoader(
            VehicleCropDataset(test_df, build_transforms(cfg["img_size"], cfg["strong_augment"], train=False)),
            batch_size=cfg["batch_size"],
            shuffle=False,
            num_workers=NUM_WORKERS,
        )
        test_metrics = evaluate_torchvision_model(model, test_loader, DEVICE)
        artifact = run_dir / "best.pt"
        latency = measure_torchvision_latency(model, test_df, cfg)
        latency["infer_backend"] = "torch"
    elif cfg.get("backbone") == "vixid_weights":
        if cfg["name"] == "vixid_vanilla" or int(cfg.get("epochs", 0)) == 0:
            artifact, val_metrics, extra = evaluate_stage2_as_is(val_df, run_dir)
            scorer = Stage2YOLODetectorScorer(weights_path=artifact)
            test_metrics = evaluate_stage2_scorer(scorer, test_df)
            (run_dir / "test_classification_report.txt").write_text(test_metrics.report)
            latency = measure_stage2_latency(scorer, test_df)
        else:
            artifact, val_metrics, extra = train_stage2_finetune_experiment(train_df, val_df, cfg, run_dir)
            scorer = Stage2YOLODetectorScorer(weights_path=artifact)
            test_metrics = evaluate_stage2_scorer(scorer, test_df)
            (run_dir / "test_classification_report.txt").write_text(test_metrics.report)
            extra["yolo_weights"] = str(artifact)
            latency = measure_stage2_latency(scorer, test_df)
    elif cfg["family"] == "yolo_cls":
        artifact, val_metrics, extra = train_yolo_cls_experiment(train_df, val_df, cfg, run_dir)
        yolo_model = YOLO(str(artifact))
        y_true, y_pred, y_prob_ev = [], [], []
        for _, row in test_df.iterrows():
            result = yolo_model.predict(str(row["image_path"]), verbose=False)[0]
            probs = result.probs
            names = yolo_model.names
            pred_idx = int(probs.top1)
            pred_label = str(names[pred_idx] if isinstance(names, dict) else names[pred_idx]).lower()
            y_true.append(int(row["label_idx"]))
            y_pred.append(CLASS_TO_IDX.get(pred_label, pred_idx))
            prob_vec = probs.data.cpu().numpy()
            y_prob_ev.append(float(prob_vec[CLASS_TO_IDX["ev"]]))
        test_metrics = _metrics_from_arrays(y_true, y_pred, y_prob_ev)
        extra["yolo_weights"] = str(artifact)
        latency = measure_yolo_cls_latency(yolo_model, test_df)
        latency["infer_backend"] = "ultralytics_cls"
    elif cfg["family"] == "stage2_yolo_det":
        artifact, val_metrics, extra = evaluate_stage2_as_is(val_df, run_dir)
        scorer = Stage2YOLODetectorScorer(weights_path=artifact)
        test_metrics = evaluate_stage2_scorer(scorer, test_df)
        (run_dir / "test_classification_report.txt").write_text(test_metrics.report)
        latency = measure_stage2_latency(scorer, test_df)
    elif cfg["family"] == "stage2_yolo_det_finetune":
        artifact, val_metrics, extra = train_stage2_finetune_experiment(train_df, val_df, cfg, run_dir)
        scorer = Stage2YOLODetectorScorer(weights_path=artifact)
        test_metrics = evaluate_stage2_scorer(scorer, test_df)
        (run_dir / "test_classification_report.txt").write_text(test_metrics.report)
        extra["yolo_weights"] = str(artifact)
        latency = measure_stage2_latency(scorer, test_df)
    else:
        raise ValueError(f"Unknown family: {cfg['family']}")

    result = {
        "name": cfg["name"],
        "family": cfg["family"],
        "backbone": cfg["backbone"],
        "run_dir": str(run_dir),
        "artifact": str(artifact),
        "val_accuracy": val_metrics.accuracy,
        "val_precision_ev": val_metrics.precision_ev,
        "val_recall_ev": val_metrics.recall_ev,
        "val_f1_ev": val_metrics.f1_ev,
        "val_roc_auc": val_metrics.roc_auc,
        "test_accuracy": test_metrics.accuracy,
        "test_precision_ev": test_metrics.precision_ev,
        "test_recall_ev": test_metrics.recall_ev,
        "test_f1_ev": test_metrics.f1_ev,
        "test_roc_auc": test_metrics.roc_auc,
        "train_elapsed_sec": round(time.time() - started, 1),
        "infer_ms_mean": round(latency.get("infer_ms_mean", float("nan")), 3),
        "infer_ms_median": round(latency.get("infer_ms_median", float("nan")), 3),
        "infer_ms_p95": round(latency.get("infer_ms_p95", float("nan")), 3),
        "infer_images": latency.get("infer_images"),
        "infer_backend": latency.get("infer_backend"),
        "n_test": int(len(test_df)),
        "config": cfg,
        "extra": extra,
    }

    result["elapsed_sec"] = result["train_elapsed_sec"]

    with open(run_dir / "result.json", "w") as f:
        json.dump(result, f, indent=2)

    return result

def run_experiment_by_name(name: str, *, replace_existing: bool = True) -> dict:
    """Train/evaluate a single experiment by name and store it in experiment_results."""
    if name not in EXPERIMENT_BY_NAME:
        known = ", ".join(EXPERIMENT_BY_NAME)
        raise KeyError(f"Unknown experiment '{name}'. Known: {known}")

    cfg = EXPERIMENT_BY_NAME[name]
    print("\n" + "=" * 80)
    print(f"Running experiment: {cfg['name']} ({cfg['family']} / {cfg['backbone']})")
    if cfg.get("description"):
        print(f"  {cfg['description']}")
    set_seed(SEED)
    result = run_single_experiment(cfg, train_df, val_df, test_df)

    if replace_existing:
        experiment_results[:] = [r for r in experiment_results if r.get("name") != name]
    experiment_results.append(result)

    print(
        f"Done: test_prec_ev={result['test_precision_ev']:.3f} | "
        f"test_f1_ev={result['test_f1_ev']:.3f} | test_acc={result['test_accuracy']:.3f} | "
        f"infer_ms_median={result['infer_ms_median']:.2f} "
        f"({result.get('infer_backend')}) | train_elapsed={result['train_elapsed_sec']}s"
    )
    return result

def run_selected_experiments(names: list[str] | None = None) -> list[dict]:
    """Run SELECTED_EXPERIMENTS (or all EXPERIMENTS if names/selection is None)."""
    if names is None:
        names = SELECTED_EXPERIMENTS
    if names is None:
        names = [cfg["name"] for cfg in EXPERIMENTS]
    unknown = [n for n in names if n not in EXPERIMENT_BY_NAME]
    if unknown:
        raise KeyError(f"Unknown experiment names: {unknown}")

    print(f"Will run {len(names)} experiment(s): {names}")
    results = []
    for name in names:
        results.append(run_experiment_by_name(name, replace_existing=True))
    return results

def load_results_from_runs(run_dirs: list[str | Path] | None = None) -> list[dict]:
    """Reload saved result.json files into experiment_results (for comparison without re-training)."""
    loaded: list[dict] = []
    if run_dirs is None:
        paths = sorted(RUNS_DIR.glob("*/result.json"))
    else:
        paths = [Path(p) / "result.json" if Path(p).is_dir() else Path(p) for p in run_dirs]
    for path in paths:
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        loaded.append(data)
        experiment_results[:] = [r for r in experiment_results if r.get("name") != data.get("name")]
        experiment_results.append(data)
    print(f"Loaded {len(loaded)} result file(s). In-memory results: {len(experiment_results)}")
    return loaded

print("\nRunner ready. Next:")
print("  - Edit SELECTED_EXPERIMENTS in the config cell, then run the 'Select & run' cell")
print("  - Or call: run_experiment_by_name('resnet18_baseline')")


### 6b) Select & run experiments

Edit `SELECTED_EXPERIMENTS` in the **configuration** cell, then run this cell.




In [ ]:
load_results_from_runs()

# names_to_run = ["vixid_vanilla", "vit_b16"]
# names_to_run = ["resnet18_baseline"]
# names_to_run = list(NEW_YOLO_EXPERIMENTS)
names_to_run = SELECTED_EXPERIMENTS

if names_to_run == []:
    print(
        "SELECTED_EXPERIMENTS is empty — nothing to run.\n"
        "Uncomment model names in the config cell, set names_to_run above, "
        "or use run_experiment_by_name('...') in the next section."
    )
else:
    # experiment_results.clear()
    run_selected_experiments(names_to_run)

print(f"\nTotal results in memory: {len(experiment_results)}")
print("Names:", [r["name"] for r in experiment_results])


### 6c) Run a single model (optional)

Use this when you only want one experiment without touching the batch list.



In [ ]:
# run_experiment_by_name("vixid_vanilla")
# run_experiment_by_name("vixid_transfer_learned")
# run_experiment_by_name("vit_b16")
# run_experiment_by_name("resnet18_baseline")
# run_experiment_by_name("yolo11n_cls")
# run_experiment_by_name("yolo26n_cls")

print("Uncomment a run_experiment_by_name(...) call above, or skip this cell.")
print(f"Results in memory: {[r['name'] for r in experiment_results]}")


### 6d) Compare results

Builds `comparison_df` from in-memory `experiment_results`.

After a selective new-YOLO run, prior models should already be loaded via `load_results_from_runs()` in **6b**. You can also reload them in the next cell.

Latency columns are **true inference** timings from the runner (`infer_ms_*`), not training wall-time.
`train_elapsed_sec` is kept separately for how long training/eval setup took.


In [ ]:
def build_comparison_df(results: list[dict] | None = None) -> pd.DataFrame:
    results = results if results is not None else experiment_results
    if not results:
        raise ValueError("No experiment_results yet. Run at least one model first.")

    df = pd.DataFrame(results).copy()
    if "train_elapsed_sec" not in df.columns and "elapsed_sec" in df.columns:
        df["train_elapsed_sec"] = df["elapsed_sec"]

    display_cols = [
        "name",
        "family",
        "backbone",
        "test_precision_ev",
        "val_precision_ev",
        "test_recall_ev",
        "test_f1_ev",
        "test_accuracy",
        "test_roc_auc",
        "infer_ms_median",
        "infer_ms_mean",
        "infer_ms_p95",
        "infer_backend",
        "train_elapsed_sec",
        "run_dir",
    ]
    display_cols = [c for c in display_cols if c in df.columns]
    df = df[display_cols].sort_values("test_f1_ev", ascending=False).reset_index(drop=True)
    return df

comparison_df = build_comparison_df()
comparison_df


In [ ]:
load_results_from_runs()
comparison_df = build_comparison_df()

print(
    f"Comparing {len(comparison_df)} model(s). "
)
print(
    "Latency note: infer_ms_* is measured with warmup + timed score/forward on test crops.\n"
    "It excludes training. Stage-2 as-is may use ONNX while fine-tune uses .pt — "
    "check infer_backend. Compare .pt vs .pt (or ONNX vs ONNX) for a fair architecture timing."
)
comparison_df[["name", "infer_backend", "infer_ms_median", "infer_ms_mean", "train_elapsed_sec"]]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_df = comparison_df.sort_values("test_f1_ev")
axes[0].barh(plot_df["name"], plot_df["test_f1_ev"], color="#7b2cbf")
axes[0].set_xlabel("Test F1 (EV class)")
axes[0].set_title("Ranked by EV F1")
axes[0].set_xlim(0, 1)

if "infer_ms_median" in comparison_df.columns and comparison_df["infer_ms_median"].notna().any():
    lat_df = comparison_df.sort_values("infer_ms_median")
    axes[1].barh(lat_df["name"], lat_df["infer_ms_median"], color="#4c6ef5")
    axes[1].set_xlabel("Median inference latency (ms / image)")
    axes[1].set_title("Inference latency (excludes training)")
else:
    axes[1].text(0.5, 0.5, "No latency data", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

summary_path = RUNS_DIR / f"comparison_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.csv"
comparison_df.to_csv(summary_path, index=False)
print(f"Saved comparison table → {summary_path}")


## 7) Export Best Model for Dashcam Pipeline

Exports the **overall best** checkpoint (by `SELECTION_METRIC`, currently `test_f1_ev`) plus `metadata.json` for `ev_ice_classifier.LocalEVClassifier`.

Supports both **torchvision** and **YOLO-cls** winners. The wrapper auto-detects `family` from metadata and exposes the same `score_crop(crop_bgr)` interface used by `consolidated_dashcam_pipeline.ipynb`.


In [ ]:
metric_col = {
    "precision_ev": "test_precision_ev",
    "f1_ev": "test_f1_ev",
    "accuracy": "test_accuracy",
    "recall_ev": "test_recall_ev",
}.get(SELECTION_METRIC, "test_precision_ev")

exportable = [
    r for r in experiment_results
    if r.get("family") in {"torchvision", "yolo_cls"}
    and r.get("backbone") != "vixid_weights"
]
if not exportable:
    raise ValueError("No exportable torchvision/yolo_cls results found in experiment_results.")

best_result = max(exportable, key=lambda r: r.get(metric_col, 0.0))
export_result = best_result
print(
    f"Best by {metric_col}: {best_result['name']} = {best_result[metric_col]:.4f} "
    f"({best_result['family']} / {best_result['backbone']})"
)

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

src_weights = Path(export_result["artifact"])
if not src_weights.exists():
    raise FileNotFoundError(f"Missing export artifact: {src_weights}")
shutil.copy2(src_weights, EXPORT_DIR / "best.pt")

metadata = {
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "selected_by_metric": metric_col,
    "best_overall_experiment": best_result["name"],
    "exported_experiment": export_result["name"],
    "family": export_result["family"],
    "backbone": export_result["backbone"],
    "class_to_idx": CLASS_TO_IDX,
    "ev_index": EV_INDEX,
    "img_size": export_result["config"]["img_size"],
    "metrics": {
        "test_precision_ev": export_result["test_precision_ev"],
        "val_precision_ev": export_result.get("val_precision_ev"),
        "test_f1_ev": export_result["test_f1_ev"],
        "test_accuracy": export_result["test_accuracy"],
    },
    "integration": {
        "class": "LocalEVClassifier",
        "module": "ev_ice_classifier.py",
        "method": "score_crop(crop_bgr) -> (ev_confidence, 'ev'|None)",
        "dashcam_flag": "consolidated_dashcam_pipeline_best_proprietary_model.ipynb",
    },
}
(EXPORT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))

if EXPORT_ONNX and export_result["family"] == "torchvision":
    model = build_torchvision_model(export_result["backbone"]).to(DEVICE)
    state = torch.load(EXPORT_DIR / "best.pt", map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()

    dummy = torch.randn(
        1, 3, export_result["config"]["img_size"], export_result["config"]["img_size"], device=DEVICE
    )
    onnx_path = EXPORT_DIR / "model.onnx"
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=17,
    )
    print(f"Exported ONNX → {onnx_path}")
elif EXPORT_ONNX and export_result["family"] == "yolo_cls":
    try:
        yolo_export = YOLO(str(EXPORT_DIR / "best.pt"))
        exported = yolo_export.export(format="onnx", imgsz=export_result["config"]["img_size"])
        print(f"Exported ONNX → {exported}")
    except Exception as exc:
        print(f"YOLO ONNX export skipped ({exc}). Pipeline uses best.pt via LocalEVClassifier.")

print(f"Exported classifier → {EXPORT_DIR}")
print(json.dumps(metadata, indent=2))


## 8) Pipeline Integration Smoke Test

Validates the exported model through the same `score_crop` interface used by `consolidated_dashcam_pipeline.ipynb`.

This cell **reloads** `ev_ice_classifier.py` so a long-running kernel picks up YOLO-cls support after export (avoids `Unsupported torchvision backbone: yolo26s-cls.pt`).


In [ ]:
import importlib
import json
import sys

if str(TRAINER_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINER_ROOT))

if "ev_ice_classifier" in sys.modules:
    del sys.modules["ev_ice_classifier"]

import ev_ice_classifier

importlib.reload(ev_ice_classifier)
from ev_ice_classifier import LocalEVClassifier

meta_path = EXPORT_DIR / "metadata.json"
weights_path = EXPORT_DIR / "best.pt"
if not meta_path.exists() or not weights_path.exists():
    raise FileNotFoundError(
        f"Missing export under {EXPORT_DIR}. Run the Export Best Model cell first."
    )

metadata = json.loads(meta_path.read_text())
print(
    f"Export metadata: experiment={metadata.get('exported_experiment')} "
    f"family={metadata.get('family')} backbone={metadata.get('backbone')} "
    f"test_f1_ev={metadata.get('metrics', {}).get('test_f1_ev')}"
)
print(f"LocalEVClassifier file: {ev_ice_classifier.__file__}")
print(f"Supports YOLO-cls: {hasattr(LocalEVClassifier, '_load_yolo_cls')}")
if not hasattr(LocalEVClassifier, "_load_yolo_cls"):
    raise RuntimeError(
        "Notebook is still using an old LocalEVClassifier without YOLO-cls support. "
        "Save/reload this notebook from disk, restart the kernel, then re-run this cell."
    )

classifier = LocalEVClassifier(model_dir=EXPORT_DIR, confidence_threshold=0.70)
print(
    f"Smoke backend: family={classifier.family} backbone={classifier.backbone} "
    f"ev_index={classifier.ev_index}"
)

smoke_df = labeled_df[labeled_df["label"].isin(VALID_LABELS)].copy()
if smoke_df.empty:
    raise ValueError("No ev/ice rows available for smoke test.")

sample_rows = smoke_df.sample(min(6, len(smoke_df)), random_state=SEED)
for _, row in sample_rows.iterrows():
    crop = cv2.imread(str(row["image_path"]))
    if crop is None:
        print(f"{row['relative_path']:40s} | true={row['label']:3s} | FAILED to read image")
        continue
    ev_conf, ev_label = classifier.score_crop(crop)
    print(
        f"{row['relative_path']:40s} | true={row['label']:3s} | "
        f"pred_ev_conf={ev_conf:.3f} | pipeline_label={ev_label}"
    )


## 9) Optional: K-Fold Cross-Validation

Set `K_FOLD > 1` in the configuration cell to estimate performance more robustly on small labeled sets. Folds are ranked by **EV precision**.


In [ ]:
if K_FOLD and K_FOLD > 1:
    cv_rows: list[dict] = []
    skf = StratifiedKFold(n_splits=K_FOLD, shuffle=True, random_state=SEED)

    for cfg in EXPERIMENTS:
        if cfg["family"] != "torchvision":
            print(f"Skipping CV for non-torchvision experiment: {cfg['name']}")
            continue

        fold_scores: list[float] = []
        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(labeled_df, labeled_df["label_idx"])):
            fold_train = labeled_df.iloc[train_idx].reset_index(drop=True)
            fold_val = labeled_df.iloc[val_idx].reset_index(drop=True)
            run_dir = RUNS_DIR / f"cv_{cfg['name']}_fold{fold_idx+1}"
            set_seed(SEED + fold_idx)
            _, metrics, _ = train_torchvision_experiment(fold_train, fold_val, cfg, run_dir)
            fold_scores.append(metrics.precision_ev)
            print(f"{cfg['name']} fold {fold_idx+1}: prec_ev={metrics.precision_ev:.3f} | f1_ev={metrics.f1_ev:.3f}")

        cv_rows.append(
            {
                "name": cfg["name"],
                "cv_folds": K_FOLD,
                "cv_precision_ev_mean": float(np.mean(fold_scores)),
                "cv_precision_ev_std": float(np.std(fold_scores)),
            }
        )

    cv_df = pd.DataFrame(cv_rows).sort_values("cv_precision_ev_mean", ascending=False)
    display(cv_df)
else:
    print("K_FOLD disabled. Set K_FOLD > 1 in config to run cross-validation.")


## 10) Significance vs `vixid_vanilla` baseline

Re-score every exported checkpoint on the **same held-out test split**, then compare each model to `vixid_vanilla` with:

1. **Paired bootstrap** (primary) — resample test images with replacement (`N_BOOTSTRAP=2000`); report mean Δ, 95% CI, and two-sided bootstrap *p*
2. **Bootstrap-SE *t*-test** — \( t = \Delta / \mathrm{SE}_{\mathrm{boot}}(\Delta) \), two-sided *p* from Student-\(t\) with \(n_{\mathrm{test}}-1\) df
3. **McNemar’s test** — paired correct/incorrect predictions (**accuracy** only)

Metrics: `accuracy`, `precision_ev`, `recall_ev`, `f1_ev`. Requires prior artifacts under `runs/*/result.json`.


In [ ]:
from __future__ import annotations

from collections import defaultdict

from scipy import stats
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

BASELINE_NAME = "vixid_vanilla"
N_BOOTSTRAP = 2000
ALPHA = 0.05
SIG_METRICS = ("accuracy", "precision_ev", "recall_ev", "f1_ev")

def _latest_results_by_name(results: list[dict] | None = None) -> dict[str, dict]:
    """Prefer in-memory results; otherwise load latest run per experiment name."""
    if results is None:
        if "experiment_results" in globals() and experiment_results:
            results = list(experiment_results)
        else:
            load_results_from_runs()
            results = list(experiment_results)

    by_name: dict[str, dict] = {}
    for r in sorted(results, key=lambda x: str(x.get("run_dir", ""))):
        name = r.get("name")
        if name:
            by_name[name] = r
    return by_name

def _metric_fns():
    return {
        "accuracy": lambda yt, yp: float(accuracy_score(yt, yp)),
        "precision_ev": lambda yt, yp: float(
            precision_score(yt, yp, pos_label=EV_INDEX, zero_division=0)
        ),
        "recall_ev": lambda yt, yp: float(
            recall_score(yt, yp, pos_label=EV_INDEX, zero_division=0)
        ),
        "f1_ev": lambda yt, yp: float(f1_score(yt, yp, pos_label=EV_INDEX, zero_division=0)),
    }

def score_model_on_frame(result: dict, frame: pd.DataFrame) -> EvalMetrics:
    """Re-run inference with the saved artifact on a shared evaluation frame."""
    name = result["name"]
    family = result["family"]
    backbone = result["backbone"]
    artifact = Path(result["artifact"])
    cfg = result.get("config") or EXPERIMENT_BY_NAME.get(name, {})

    if not artifact.exists():
        raise FileNotFoundError(f"{name}: artifact missing at {artifact}")

    if backbone == "vixid_weights" or name.startswith("vixid_"):
        scorer = Stage2YOLODetectorScorer(weights_path=artifact)
        return evaluate_stage2_scorer(scorer, frame)

    if family == "torchvision":
        model = build_torchvision_model(backbone).to(DEVICE)
        state = torch.load(artifact, map_location=DEVICE)
        model.load_state_dict(state)
        model.eval()
        loader = DataLoader(
            VehicleCropDataset(
                frame,
                transform=build_transforms(cfg.get("img_size", 224), False, train=False),
            ),
            batch_size=int(cfg.get("batch_size", 16)),
            shuffle=False,
            num_workers=NUM_WORKERS,
        )
        return evaluate_torchvision_model(model, loader, DEVICE)

    if family == "yolo_cls":
        yolo_model = YOLO(str(artifact))
        y_true: list[int] = []
        y_pred: list[int] = []
        y_prob_ev: list[float] = []
        for _, row in frame.iterrows():
            pred = yolo_model.predict(str(row["image_path"]), verbose=False)[0]
            probs = pred.probs
            if probs is None:
                raise RuntimeError(f"{name}: YOLO-cls returned no probs for {row['image_path']}")
            prob_vec = probs.data.detach().cpu().numpy()
            pred_idx = int(prob_vec.argmax())
            names = yolo_model.names
            pred_label = str(names[pred_idx] if isinstance(names, dict) else names[pred_idx]).lower()
            y_true.append(int(row["label_idx"]))
            y_pred.append(CLASS_TO_IDX.get(pred_label, pred_idx))
            y_prob_ev.append(float(prob_vec[CLASS_TO_IDX["ev"]]))
        return _metrics_from_arrays(y_true, y_pred, y_prob_ev)

    raise ValueError(f"Unsupported family/backbone for significance scoring: {family=} {backbone=}")

def paired_bootstrap_tests(
    y_true: np.ndarray,
    y_pred_base: np.ndarray,
    y_pred_model: np.ndarray,
    n_boot: int = N_BOOTSTRAP,
    seed: int = SEED,
) -> dict[str, dict]:
    """Paired bootstrap Δ = model - baseline for each metric + t-test on Δs."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    metric_fns = _metric_fns()
    out: dict[str, dict] = {}

    for metric_name, fn in metric_fns.items():
        base_point = fn(y_true, y_pred_base)
        model_point = fn(y_true, y_pred_model)
        diffs = np.empty(n_boot, dtype=np.float64)
        for b in range(n_boot):
            idx = rng.integers(0, n, size=n)
            diffs[b] = fn(y_true[idx], y_pred_model[idx]) - fn(y_true[idx], y_pred_base[idx])

        p_boot = float(2.0 * min(np.mean(diffs <= 0.0), np.mean(diffs >= 0.0)))
        p_boot = min(p_boot, 1.0)
        ci_lo, ci_hi = np.quantile(diffs, [0.025, 0.975])

        delta = model_point - base_point
        se = float(diffs.std(ddof=1))
        if se == 0.0:
            t_stat = 0.0
            p_t = 1.0
        else:
            t_stat = float(delta / se)
            p_t = float(2.0 * stats.t.sf(abs(t_stat), df=max(n - 1, 1)))

        out[metric_name] = {
            "baseline": base_point,
            "model": model_point,
            "delta": delta,
            "boot_delta_mean": float(diffs.mean()),
            "boot_delta_std": se,
            "ci95_lo": float(ci_lo),
            "ci95_hi": float(ci_hi),
            "p_bootstrap": p_boot,
            "t_stat": t_stat,
            "p_ttest": p_t,
            "significant_bootstrap": bool(p_boot < ALPHA),
            "significant_ttest": bool(p_t < ALPHA),
        }
    return out

def mcnemar_pvalue(y_true: np.ndarray, y_pred_base: np.ndarray, y_pred_model: np.ndarray) -> dict:
    """McNemar test on paired correctness (continuity-corrected chi-square)."""
    base_ok = y_pred_base == y_true
    model_ok = y_pred_model == y_true
    b = int(np.sum(~base_ok & model_ok))
    c = int(np.sum(base_ok & ~model_ok))
    if b + c == 0:
        p = 1.0
        stat = 0.0
    else:
        stat = (abs(b - c) - 1.0) ** 2 / (b + c)
        p = float(stats.chi2.sf(stat, df=1))
    return {
        "n_model_only_correct": b,
        "n_baseline_only_correct": c,
        "mcnemar_stat": float(stat),
        "p_mcnemar": p,
        "significant_mcnemar": bool(p < ALPHA),
    }

set_seed(SEED)
if "test_df" not in globals():
    train_df, val_df, test_df = split_dataframe(labeled_df)

results_by_name = _latest_results_by_name()
if BASELINE_NAME not in results_by_name:
    raise KeyError(
        f"Baseline '{BASELINE_NAME}' not found in results. "
        f"Have: {sorted(results_by_name)}"
    )

print(f"Scoring {len(results_by_name)} model(s) on test_df (n={len(test_df)}) ...")
print(f"Baseline: {BASELINE_NAME}")
print(f"Bootstrap samples: {N_BOOTSTRAP} | alpha={ALPHA}\n")

pred_cache: dict[str, EvalMetrics] = {}
pred_dir = RUNS_DIR / "_significance_preds"
pred_dir.mkdir(parents=True, exist_ok=True)
for name, result in results_by_name.items():
    pred_path = pred_dir / f"{name}_test_preds.json"
    print(f"  infer: {name} ({result['family']} / {result['backbone']})")
    metrics = score_model_on_frame(result, test_df)
    pred_cache[name] = metrics
    pred_path.write_text(
        json.dumps(
            {
                "name": name,
                "y_true": metrics.y_true,
                "y_pred": metrics.y_pred,
                "y_prob_ev": metrics.y_prob_ev,
                "accuracy": metrics.accuracy,
                "precision_ev": metrics.precision_ev,
                "recall_ev": metrics.recall_ev,
                "f1_ev": metrics.f1_ev,
            }
        )
    )

y_true = np.asarray(pred_cache[BASELINE_NAME].y_true, dtype=int)
for name, metrics in pred_cache.items():
    yt = np.asarray(metrics.y_true, dtype=int)
    if not np.array_equal(yt, y_true):
        raise RuntimeError(f"Label mismatch vs baseline for {name}")

y_pred_base = np.asarray(pred_cache[BASELINE_NAME].y_pred, dtype=int)

sig_rows: list[dict] = []
for name, metrics in sorted(pred_cache.items()):
    if name == BASELINE_NAME:
        continue
    y_pred = np.asarray(metrics.y_pred, dtype=int)
    boot = paired_bootstrap_tests(y_true, y_pred_base, y_pred)
    mcn = mcnemar_pvalue(y_true, y_pred_base, y_pred)
    for metric_name, stats_row in boot.items():
        sig_rows.append(
            {
                "model": name,
                "metric": metric_name,
                "baseline_value": stats_row["baseline"],
                "model_value": stats_row["model"],
                "delta": stats_row["delta"],
                "boot_delta_mean": stats_row["boot_delta_mean"],
                "ci95_lo": stats_row["ci95_lo"],
                "ci95_hi": stats_row["ci95_hi"],
                "p_bootstrap": stats_row["p_bootstrap"],
                "t_stat": stats_row["t_stat"],
                "p_ttest": stats_row["p_ttest"],
                "significant_ttest": stats_row["significant_ttest"],
                "significant_bootstrap": stats_row["significant_bootstrap"],
                "p_mcnemar": mcn["p_mcnemar"] if metric_name == "accuracy" else np.nan,
                "significant_mcnemar": mcn["significant_mcnemar"] if metric_name == "accuracy" else False,
                "n_model_only_correct": mcn["n_model_only_correct"] if metric_name == "accuracy" else np.nan,
                "n_baseline_only_correct": mcn["n_baseline_only_correct"] if metric_name == "accuracy" else np.nan,
            }
        )

significance_df = (
    pd.DataFrame(sig_rows)
    .sort_values(["metric", "p_ttest", "delta"], ascending=[True, True, False])
    .reset_index(drop=True)
)

sig_path = RUNS_DIR / f"significance_vs_{BASELINE_NAME}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.csv"
significance_df.to_csv(sig_path, index=False)
print(f"\nSaved → {sig_path}")

f1_df = significance_df[significance_df["metric"] == "f1_ev"].copy()
f1_df = f1_df.sort_values("delta", ascending=False).reset_index(drop=True)
print("\n=== test_f1_ev vs vixid_vanilla (paired bootstrap + t-test on Δ) ===")
display(
    f1_df[
        [
            "model",
            "baseline_value",
            "model_value",
            "delta",
            "ci95_lo",
            "ci95_hi",
            "p_bootstrap",
            "p_ttest",
            "significant_ttest",
        ]
    ]
)

acc_df = significance_df[significance_df["metric"] == "accuracy"].copy()
acc_df = acc_df.sort_values("delta", ascending=False).reset_index(drop=True)
print("\n=== accuracy vs vixid_vanilla (+ McNemar) ===")
display(
    acc_df[
        [
            "model",
            "baseline_value",
            "model_value",
            "delta",
            "p_bootstrap",
            "p_ttest",
            "p_mcnemar",
            "significant_ttest",
            "significant_mcnemar",
        ]
    ]
)

print("\nFull metric table:")
display(significance_df)


In [ ]:
plot_df = significance_df[significance_df["metric"] == "f1_ev"].sort_values("delta")
fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(plot_df) + 1)))
colors = ["#2a9d8f" if p < ALPHA else "#6c757d" for p in plot_df["p_ttest"]]
ax.barh(
    plot_df["model"],
    plot_df["delta"],
    xerr=np.vstack([
        plot_df["delta"] - plot_df["ci95_lo"],
        plot_df["ci95_hi"] - plot_df["delta"],
    ]),
    color=colors,
    alpha=0.85,
    capsize=3,
)
ax.axvline(0.0, color="black", linewidth=1)
ax.set_xlabel(f"Δ test_f1_ev vs {BASELINE_NAME}")
ax.set_title(f"Paired bootstrap Δ F1 (teal = t-test p < {ALPHA})")
plt.tight_layout()
plt.show()

n_sig = int(significance_df.query("metric == 'f1_ev' and significant_ttest").shape[0])
print(f"Models with significant F1 improvement/change vs baseline (t-test p < {ALPHA}): {n_sig}")
